# Lab 5 · Red de incidencias, antenas, zonas y clientes afectados

Notebook preparado para **AWS Academy Learner Lab / SageMaker Notebook**. El objetivo es representar la red telco como un grafo, calcular métricas estructurales, estimar clientes afectados por la caída de nodos y cruzar la topología con incidencias.

Los CSV sintéticos incluidos respetan la estructura del laboratorio: `network_nodes.csv`, `network_edges.csv`, `customer_nodes.csv` y `network_incidents.csv`.

In [ ]:
# Instalación de dependencias básicas. En SageMaker normalmente ya están instaladas.
%pip -q install pandas numpy matplotlib networkx scikit-learn s3fs

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Por defecto se leen los CSV incluidos en la carpeta local data/.
# Si prefieres leer desde S3, cambia USE_S3=True y ajusta S3_PREFIX.
USE_S3 = False
DATA_DIR = Path('data')
S3_PREFIX = 's3://TU_BUCKET/TU_CARPETA'  # ejemplo: s3://mi-bucket/graph

def data_path(filename):
    return f"{S3_PREFIX}/{filename}" if USE_S3 else str(DATA_DIR / filename)

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

import networkx as nx

## Parte A. Cargar datos y construir el grafo

In [ ]:
nodes = pd.read_csv(data_path('network_nodes.csv'))
edges = pd.read_csv(data_path('network_edges.csv'))
cust = pd.read_csv(data_path('customer_nodes.csv'))
inc = pd.read_csv(data_path('network_incidents.csv'))

print('Nodos:', nodes.shape)
print('Enlaces:', edges.shape)
print('Clientes:', cust.shape)
print('Incidencias:', inc.shape)
display(nodes.head())
display(edges.head())

In [ ]:
G = nx.Graph()
for _, r in nodes.iterrows():
    G.add_node(r['node_id'], node_type=r['node_type'], region=r['region'],
               zone=r['zone'], capacity=r['capacity_gbps'], lat=r['lat'], lon=r['lon'])
for _, r in edges.iterrows():
    G.add_edge(r['source'], r['target'], link_type=r['link_type'],
               capacity=r['capacity_gbps'], distance=r['distance_km'])

clientes_por_nodo = cust.groupby('access_node_id').size()
for nid in G.nodes():
    G.nodes[nid]['n_clientes'] = int(clientes_por_nodo.get(nid, 0))

print('Nodos en el grafo:', G.number_of_nodes())
print('Enlaces en el grafo:', G.number_of_edges())
print('¿Grafo conexo?:', nx.is_connected(G))
print('Clientes totales:', sum(nx.get_node_attributes(G, 'n_clientes').values()))

## Parte B. Métricas estructurales

Calculamos grado, centralidad de intermediación, cercanía y puntos de articulación.

In [ ]:
grado = dict(G.degree())
bet = nx.betweenness_centrality(G)
clo = nx.closeness_centrality(G)

metricas = pd.DataFrame({
    'node_id': list(G.nodes()),
    'tipo': [G.nodes[n]['node_type'] for n in G.nodes()],
    'region': [G.nodes[n]['region'] for n in G.nodes()],
    'grado': [grado[n] for n in G.nodes()],
    'betweenness': [bet[n] for n in G.nodes()],
    'closeness': [clo[n] for n in G.nodes()],
    'clientes_directos': [G.nodes[n]['n_clientes'] for n in G.nodes()],
}).sort_values('betweenness', ascending=False)

display(metricas.head(12))

In [ ]:
articulacion = list(nx.articulation_points(G))
print('Puntos de articulación:', len(articulacion))
for nid in articulacion:
    print(f"{nid:12s} | {G.nodes[nid]['node_type']:12s} | clientes directos: {G.nodes[nid]['n_clientes']}")

## Parte C. Clientes afectados si cae un nodo

La función elimina temporalmente un nodo y comprueba qué nodos de acceso quedan desconectados del núcleo.

In [ ]:
def clientes_afectados_si_cae(G, nodo):
    H = G.copy()
    clientes_directos = G.nodes[nodo].get('n_clientes', 0)
    H.remove_node(nodo)
    cores = [n for n in H.nodes() if H.nodes[n]['node_type'] == 'core']
    afectados = clientes_directos
    for nid, attr in H.nodes(data=True):
        if attr.get('n_clientes', 0) > 0:
            conectado = any(nx.has_path(H, nid, c) for c in cores) if cores else False
            if not conectado:
                afectados += attr['n_clientes']
    return int(afectados)

impacto = {n: clientes_afectados_si_cae(G, n) for n in G.nodes()}
resumen = metricas.set_index('node_id').copy()
resumen['clientes_afectados'] = pd.Series(impacto)
resumen = resumen.sort_values('clientes_afectados', ascending=False)
display(resumen.head(15))
resumen.to_csv(OUTPUT_DIR/'lab05_criticidad_nodos.csv')

## Parte D. Incidencias e impacto cliente × minuto

In [ ]:
inc_nodo = inc.merge(nodes[['node_id','node_type','region','zone']], on='node_id', how='left')
inc_nodo['clientes_afectados'] = inc_nodo['node_id'].map(impacto)
inc_nodo['impacto_cliente_minuto'] = inc_nodo['clientes_afectados'] * inc_nodo['duration_minutes']

display(inc_nodo.groupby(['node_type','severity']).size().rename('incidencias').reset_index())
display(inc_nodo.sort_values('impacto_cliente_minuto', ascending=False).head(10))

resumen_region = inc_nodo.groupby('region').agg(
    incidencias=('incident_id','count'),
    minutos_totales=('duration_minutes','sum'),
    impacto=('impacto_cliente_minuto','sum')
).sort_values('impacto', ascending=False)
display(resumen_region)
inc_nodo.to_csv(OUTPUT_DIR/'lab05_incidencias_con_impacto.csv', index=False)

## Parte E. Visualización de la red

In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.5)
colors_by_type = {'core':'tab:red', 'aggregation':'tab:orange', 'access':'tab:blue'}
node_colors = [colors_by_type[G.nodes[n]['node_type']] for n in G.nodes()]
node_sizes = [120 + G.nodes[n]['n_clientes'] * 5 for n in G.nodes()]
labels = {n:n for n in G.nodes() if G.nodes[n]['node_type'] in ['core','aggregation']}

plt.figure(figsize=(13,9))
nx.draw_networkx_edges(G, pos, alpha=0.30)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes)
nx.draw_networkx_labels(G, pos, labels, font_size=8)
plt.title('Red de la operadora: núcleo, agregación y acceso')
plt.axis('off'); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab05_red_por_tipo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
valores_bet = [bet[n] for n in G.nodes()]
plt.figure(figsize=(13,9))
nx.draw_networkx_edges(G, pos, alpha=0.30)
nodos = nx.draw_networkx_nodes(G, pos, node_color=valores_bet, cmap='YlOrRd', node_size=node_sizes)
plt.colorbar(nodos, label='Centralidad de intermediación')
nx.draw_networkx_labels(G, pos, labels, font_size=8)
plt.title('Criticidad estructural de la red')
plt.axis('off'); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab05_criticidad_red.png', dpi=150, bbox_inches='tight')
plt.show()

## Cierre

Ficheros generados en `outputs/`: tabla de criticidad, incidencias con impacto y dos imágenes para el entregable.